In [1]:
input_file = "datasets/to-live-a-novel-cleaned.txt"
with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()
print(f"text length: {len(text)}")

text length: 98634


In [2]:
# 获取 text 中所有不重复的字符（集合会自动去重）
unique_chars_set = set(text)

# 对列表中的字符进行排序，并转成数组，这样就有索引可以用了
chars_list = sorted(list(unique_chars_set))

# 用字典存储 字符 -> 索引 的映射
char2idx = {}
for i in range(len(chars_list)):
    c = chars_list[i]
    # 将字符作为键，索引作为值存入字典
    char2idx[c] = i

# 再来一个字典存储 索引 -> 字符 的映射
idx2char = {}
for i in range(len(chars_list)):
    c = chars_list[i]
    # 将索引作为键，字符作为值存入字典
    idx2char[i] = c

vocab_size = len(chars_list)
print(f"词汇表大小: {vocab_size}")

print(f"字符 '我' 对应的索引是: {char2idx['我']}")
print(f"索引 '20' 对应的字符是: {idx2char[20]}")


词汇表大小: 1863
字符 '我' 对应的索引是: 725
索引 '20' 对应的字符是: 丈


In [3]:
def one_hot_encode(index, vocab_size):
    """
    将整数索引编码为 one-hot 向量
    """
    # 初始化一个长度为词汇表大小、全为 0 的列表
    one_hot_vector = [0] * vocab_size
    
    # 检查索引是否越界
    if 0 <= index < vocab_size:
        # 将对应索引的位置设为 1
        one_hot_vector[index] = 1
    else:
        print(f"警告：索引 {index} 超出词汇表范围 (0 ~ {vocab_size-1})")
        
    return one_hot_vector

def one_hot_decode(one_hot_vector):
    """
    将 one-hot 向量解码为整数索引
    """
    # 找到列表中值为 1 的元素的索引
    if 1 in one_hot_vector:
        return one_hot_vector.index(1)
    else:
        return -1  # 返回 -1 表示无效向量

In [4]:
# 测试一下「我」这个字的编码和还原
test_char = "我"
print(f"=== 测试字符: '{test_char}' ===")

# 字符 -> 整数索引
char_idx = char2idx.get(test_char)
print(f"[char2idx] 字符 '{test_char}' 对应的整数索引为: {char_idx}")

# 整数索引 -> one-hot 编码
if char_idx is not None:
    one_hot_vec = one_hot_encode(char_idx, vocab_size)
    print(f"[one_hot_encode] 生成的 One-hot 向量长度为: {len(one_hot_vec)}")
    print(f"[one_hot_encode] 向量中值为 1 的位置在: {one_hot_vec.index(1)}")

    # one-hot 解码 -> 整数索引 (验证还原数字索引)
    decoded_idx = one_hot_decode(one_hot_vec)
    print(f"[one_hot_decode] One-hot 向量解码出的索引为: {decoded_idx}")

    # 整数索引 -> 字符 (验证最终还原汉字)
    decoded_char = idx2char.get(decoded_idx)
    print(f"[idx2char] 索引还原出的字符为: '{decoded_char}'")

    # 总结
    print(f"==> 最终验证结果: {'成功' if test_char == decoded_char else '失败'}")
else:
    print(f"错误：词汇表中不包含字符 '{test_char}'")

=== 测试字符: '我' ===
[char2idx] 字符 '我' 对应的整数索引为: 725
[one_hot_encode] 生成的 One-hot 向量长度为: 1863
[one_hot_encode] 向量中值为 1 的位置在: 725
[one_hot_decode] One-hot 向量解码出的索引为: 725
[idx2char] 索引还原出的字符为: '我'
==> 最终验证结果: 成功


In [5]:

import torch

class CharDataset(torch.utils.data.Dataset):
    def __init__(self, text: str, char2idx: dict, learn_char_len: int = 32, step_char_len: int = 1):
        self.char2idx = char2idx
        self.vocab_size = len(char2idx)  # 提取词汇表大小，供 one-hot 编码使用
        self.learn_char_len = learn_char_len

        # 将整个原始文本转换成对应的数字索引列表
        self.data = []
        for c in text:
            idx = char2idx.get(c)
            # 如果字符在词表中，加入索引列表
            if idx is not None:
                self.data.append(idx)

        # 初始化一个空列表，用于保存所有提取出的训练样本 (数字索引形式)
        self.samples = []

        # 使用滑动窗口遍历数字索引列表
        for i in range(0, len(self.data) - learn_char_len, step_char_len):
            # 获取输入序列 x 的索引
            x_idx = self.data[i : i + learn_char_len]
            # 获取目标序列 y 的索引 (x 向右偏移一位)
            y_idx = self.data[i + 1 : i + learn_char_len + 1]
            
            # 将这对索引序列保存为样本
            self.samples.append((x_idx, y_idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # 取出当前样本的索引序列
        x_idx, y_idx = self.samples[idx]
        
        # 遍历 x 中的每个整数索引，将其编码为 one-hot 向量
        x_one_hot = [one_hot_encode(i, self.vocab_size) for i in x_idx]
        # 遍历 y 中的每个整数索引，将其编码为 one-hot 向量
        y_one_hot = [one_hot_encode(i, self.vocab_size) for i in y_idx]
        
        # 转换为 torch.tensor
        x_tensor = torch.tensor(x_one_hot, dtype=torch.float32)
        y_tensor = torch.tensor(y_one_hot, dtype=torch.float32)
        
        return x_tensor, y_tensor

In [6]:
# 初始化 Dataset 和 DataLoader
char_dataset = CharDataset(text, char2idx=char2idx, learn_char_len=128)
dataloader = torch.utils.data.DataLoader(char_dataset, batch_size=64, num_workers=8, shuffle=True, pin_memory=True)

# 获取一个 batch 的数据
data_iter = iter(dataloader)
for batch_idx in range(1):
    x_batch, y_batch = next(data_iter)
    
    print(f"=== Batch {batch_idx + 1} ===")
    print(f"输入批次形状: {x_batch.shape}  (批次大小, 学习序列长度, 词汇表大小)")
    print(f"目标批次形状: {y_batch.shape}")
    
    # 为了演示清晰，这里只挑前 3 个样本进行打印
    num_samples_to_print = min(3, x_batch.size(0))
    
    for i in range(num_samples_to_print):
        # 取出单个样本的张量 (形状: [seq_len, vocab_size])
        x_sample = x_batch[i]
        y_sample = y_batch[i]
        
        # 将 x 的 one-hot 向量解码为索引，再转为字符
        x_indices = [one_hot_decode(vec.tolist()) for vec in x_sample]
        x_str = "".join([idx2char.get(idx, '?') for idx in x_indices])
        
        # 将 y 的 one-hot 向量解码为索引，再转为字符
        y_indices = [one_hot_decode(vec.tolist()) for vec in y_sample]
        y_str = "".join([idx2char.get(idx, '?') for idx in y_indices])
        
        print(f"\n样本 {i+1}:")
        print(f"  输入 x 还原的前5个索引: {x_indices[:5]}")
        print(f"  目标 y 还原的前5个索引: {y_indices[:5]}")
        print(f"  输入 x (文本): '{x_str}'")
        print(f"  目标 y (文本): '{y_str}'")
        print("-" * 50)

=== Batch 1 ===
输入批次形状: torch.Size([64, 128, 1863])  (批次大小, 学习序列长度, 词汇表大小)
目标批次形状: torch.Size([64, 128, 1863])

样本 1:
  输入 x 还原的前5个索引: [664, 1713, 1640, 551, 13]
  目标 y 还原的前5个索引: [1713, 1640, 551, 13, 2]
  输入 x (文本): '心里踏实。<EOS>"她拆拆缝缝给凤霞和有庆都做了件衣服，两个孩子穿上后看起来还很新。<EOS>后来我才知道她把自己的衣服也拆了，看到我生气，她笑了笑说："衣服不穿坏起来快。<EOS>我是不会穿它们了，可不能跟着我糟蹋了。<EOS>"家珍说也给我做一件，谁'
  目标 y (文本): '里踏实。<EOS>"她拆拆缝缝给凤霞和有庆都做了件衣服，两个孩子穿上后看起来还很新。<EOS>后来我才知道她把自己的衣服也拆了，看到我生气，她笑了笑说："衣服不穿坏起来快。<EOS>我是不会穿它们了，可不能跟着我糟蹋了。<EOS>"家珍说也给我做一件，谁知'
--------------------------------------------------

样本 2:
  输入 x 还原的前5个索引: [23, 955, 13, 2, 5]
  目标 y 还原的前5个索引: [955, 13, 2, 5, 6]
  输入 x (文本): '下来。<EOS>可我们放心不下凤霞，她和别人不一样，她老了谁会管她？<EOS>凤霞说起来又聋又哑，她也是女人，不会不知道男婚女嫁的事。<EOS>村里每年都有嫁出去娶进来的，敲锣打鼓热闹一阵，到那时候凤霞握着锄头总要看得发呆，村里几个年轻人就对凤霞指指点点，'
  目标 y (文本): '来。<EOS>可我们放心不下凤霞，她和别人不一样，她老了谁会管她？<EOS>凤霞说起来又聋又哑，她也是女人，不会不知道男婚女嫁的事。<EOS>村里每年都有嫁出去娶进来的，敲锣打鼓热闹一阵，到那时候凤霞握着锄头总要看得发呆，村里几个年轻人就对凤霞指指点点，笑'
--------------------------------------------------

样本 3:
  输入 x 还原的前5个索引: [6, 9, 3, 173,

In [7]:

class RNNCell(torch.nn.Module):
    """
    图 5-1 所示的 RNN 最基础小积木。

    输入：
        x      : 当前时刻输入，形状 (batch, vocab_size)
        h_prev : 上一时刻隐状态，形状 (batch, hidden_size)

    输出：
        y      : 当前时刻的预测（下一个字符的概率分布），形状 (batch, vocab_size)
        h      : 更新后的隐状态，形状 (batch, hidden_size)
    """
    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        # X_t · W_xh：把输入字符从 vocab_size 维映射到 hidden_size 维
        self.linear_x = torch.nn.Linear(vocab_size, hidden_size, bias=False)

        # H_{t-1} · W_hh：把上一时刻隐状态做线性变换
        self.linear_h = torch.nn.Linear(hidden_size, hidden_size, bias=False)

        # b_h：偏置项，两个线性层的偏置合并到这里统一加一次即可
        self.bias = torch.nn.Parameter(torch.zeros(hidden_size))

        # 输出层：从 hidden_size 维隐状态投影回 vocab_size 维，用于预测下一个字符
        self.linear_y = torch.nn.Linear(hidden_size, vocab_size)

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor):
        """
        式 5-2：
            H_t = tanh(X_t · W_xh + H_{t-1} · W_hh + b_h)
        再由 H_t 投影出预测 Y_t。
        """
        # 两路线性变换 + 偏置，然后过 tanh 激活
        h = torch.tanh(self.linear_x(x) + self.linear_h(h_prev) + self.bias)

        # 从隐状态投影到词汇表维度，得到下一个字符的预测分布
        y = self.linear_y(h)

        return y, h

In [8]:
class RNN(torch.nn.Module):
    """
    完整的循环神经网络。

    将 RNNCell 按时间步循环展开，处理一整个序列。

    输入：
        x : 形状 (batch, seq_len, vocab_size)，one-hot 编码的输入序列
    输出：
        outputs : 形状 (batch, seq_len, vocab_size)，每个时间步的预测分布
        h       : 最终隐状态，形状 (batch, hidden_size)
    """
    def __init__(self, vocab_size: int, hidden_size: int):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        # 复用我们刚才实现的小积木
        self.rnn_cell = RNNCell(vocab_size, hidden_size)

    def forward(self, x: torch.Tensor, h_prev: torch.Tensor = None):
        batch_size = x.size(0)
        seq_len = x.size(1)

        # 如果没有传入初始隐状态，就用全零初始化
        if h_prev is None:
            h_prev = torch.zeros(batch_size, self.hidden_size, device=x.device)

        # 用一个列表收集每个时间步的输出
        outputs = []

        # 按时间步循环 —— 这就是 RNN 中 "Recurrent" 的体现
        for t in range(seq_len):
            # 取出第 t 个时间步的输入，形状 (batch, vocab_size)
            x_t = x[:, t, :]
            # 送进 RNNCell，得到当前时刻的预测和更新后的隐状态
            y_t, h_prev = self.rnn_cell(x_t, h_prev)
            # 收集输出
            outputs.append(y_t)

        # 把列表堆叠成张量，形状 (batch, seq_len, vocab_size)
        outputs = torch.stack(outputs, dim=1)

        return outputs, h_prev

In [19]:
# 初始化 RNN 网络
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

hidden_size = 1024
model = RNN(vocab_size=vocab_size, hidden_size=hidden_size)
model = model.to(device)

def generate_text(model, char2idx, idx2char, seed_text, gen_len=100):
    """
    RNN 前向推理生成文本。

    参数：
        model        : RNN 网络
        char2idx     : 字符 -> 索引 映射
        idx2char     : 索引 -> 字符 映射
        seed_text    : 种子文本，模型以此为起点续写
        gen_len      : 要生成的字符数量
    """
    model.eval()
    vocab_size = len(char2idx)
    device = next(model.parameters()).device

    # 把种子文本转成索引序列
    seed_indices = [char2idx[c] for c in seed_text if c in char2idx]

    # 用种子文本初始化隐状态（让模型"读"一遍种子文本）
    h = None
    for idx in seed_indices:
        # 构造当前字符的 one-hot 张量，形状为 (1, 1, vocab_size)
        x_t = torch.tensor(one_hot_encode(idx, vocab_size), dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        # 送入模型，更新隐状态
        _, h = model(x_t, h)

    # 从种子文本的最后一个字符开始续写
    generated = list(seed_text)

    # 取种子最后一个字符作为起始输入
    if len(seed_indices) > 0:
        last_idx = seed_indices[-1]
    else:
        last_idx = 0

    for _ in range(gen_len):        
        # 构造当前字符的 one-hot 张量，形状为 (1, 1, vocab_size)
        x_t = torch.tensor(one_hot_encode(last_idx, vocab_size), dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        # 送入模型，得到预测分布和新的隐状态
        y_t, h = model(x_t, h)

        # 取出预测分布并计算概率
        # argmax() ：从数组里找出最大的值，返回它的索引
        next_idx = y_t.squeeze(0).squeeze(0).argmax().item()

        generated.append(idx2char[next_idx])
        last_idx = next_idx

    return "".join(generated)

seed = "凤霞命苦啊，"
text = generate_text(model, char2idx, idx2char, seed_text=seed, gen_len=80)
print(f"\n种子: '{seed}'")
print(f"生成: '{text}'")
print("-" * 50)

Using device: cuda

种子: '凤霞命苦啊，'
生成: '凤霞命苦啊，渣渣伙蜜哆受停客搬朽垃瓶部逃屑逃理够凑钻徒镰霸谣蚁权乌玩朽痕得哧达确党磨按湾哧懂霸按湾哧懂霸按湾哧懂霸按湾哧懂霸按湾哧懂霸按湾哧懂霸按湾哧懂霸按湾哧懂霸按湾哧懂'
--------------------------------------------------


In [10]:
def manual_clip_grad_norm_(parameters, max_norm: float):
    """
    手动实现的按全局 L2 范数裁剪梯度。
    """
    if isinstance(parameters, torch.Tensor):
        parameters = [parameters]
        
    # 过滤出有梯度的参数
    params_with_grad = [p for p in parameters if p.grad is not None]
    if len(params_with_grad) == 0:
        return torch.tensor(0.)
        
    # 计算所有参数梯度拼在一起的全局 L2 范数
    total_norm = torch.norm(torch.stack([torch.norm(p.grad.detach(), 2) for p in params_with_grad]), 2)
    
    # 计算缩放系数
    clip_coef = max_norm / (total_norm + 1e-6)
    
    # 如果 total_norm < max_norm，coef > 1，我们不应放大梯度，所以限制最大为 1.0
    clip_coef_clamped = torch.clamp(clip_coef, max=1.0)
    
    # 原地缩放所有梯度
    for p in params_with_grad:
        p.grad.detach().mul_(clip_coef_clamped)
        
    return total_norm

In [11]:
class manual_adam:
    """
    手动实现的 Adam 优化器。
    """
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr = lr
        self.beta1, self.beta2 = betas
        self.eps = eps
        
        # 初始化状态：一阶矩 m 和 二阶矩 v
        # zeros_like 会自动继承原参数的设备和数据类型
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0  # 时间步
        
    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()  # 清空计算图
                p.grad.zero_()    # 梯度置零
                
    def step(self):
        self.t += 1
        for i, p in enumerate(self.params):
            if p.grad is None:
                continue
                
            grad = p.grad.data
            
            # 更新一阶矩估计 m (Momentum)
            self.m[i].mul_(self.beta1).add_(grad, alpha=1 - self.beta1)
            
            # 更新二阶矩估计 v (RMSProp)
            self.v[i].mul_(self.beta2).addcmul_(grad, grad, value=1 - self.beta2)
            
            # 计算偏差校正
            m_hat = self.m[i] / (1 - self.beta1 ** self.t)
            v_hat = self.v[i] / (1 - self.beta2 ** self.t)
            
            # 更新参数 (原地操作)
            # p = p - lr * m_hat / (sqrt(v_hat) + eps)
            p.data.addcdiv_(m_hat, torch.sqrt(v_hat) + self.eps, value=-self.lr)

In [12]:
import time

learning_rate = 0.001
optimizer = manual_adam(model.parameters(), lr=learning_rate)
loss_fn = torch.nn.CrossEntropyLoss()

# 获取总 batch 数
total_batches = len(dataloader)

num_epochs = 20
for epoch in range(num_epochs):
    total_loss = 0
    batch_count = 0

    for batch_idx, (x_batch, y_batch) in enumerate(dataloader):
        batch_start_time = time.time()
        # 把数据搬到 GPU 上
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        # 前向传播
        outputs, _ = model(x_batch)

        # CrossEntropyLoss 期望的输入形状：
        #   predictions: (N, C)  —— 把 batch 和 seq_len 拍平
        #   targets:      (N,)   —— 每个位置的正确类别索引
        # 所以我们需要把 one-hot 的 y 转成类别索引
        y_indices = y_batch.argmax(dim=2)

        # 把预测和目标 reshape 拍平
        preds = outputs.reshape(-1, vocab_size)
        targets = y_indices.reshape(-1)

        loss = loss_fn(preds, targets)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        
        # 梯度裁剪
        manual_clip_grad_norm_(model.parameters(), max_norm=5.0)
        
        # Adam 更新参数
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        # 计算本 batch 耗时
        batch_time = time.time() - batch_start_time
        
        # 估计剩余时间 (ETA)
        remaining_batches = total_batches - (batch_idx + 1)
        eta = remaining_batches * batch_time
        
        # 用 \r 动态刷新打印同一行
        print(f"\rEpoch [{epoch+1}/{num_epochs}] | Batch [{batch_idx+1}/{total_batches}] | 损失: {loss.item():.4f} | 单步耗时: {batch_time:.3f}s | 预计剩余: {eta:.1f}s", end="")

    # 每个 epoch 结束后换行并打印汇总信息
    print(f"\rEpoch [{epoch+1}/{num_epochs}] 完成 | 平均损失: {total_loss / batch_count:.4f}")

Epoch [1/20] 完成 | 平均损失: 3.07360] | 损失: 1.5939 | 单步耗时: 0.088s | 预计剩余: 0.0ss
Epoch [2/20] 完成 | 平均损失: 0.94130] | 损失: 0.5105 | 单步耗时: 0.087s | 预计剩余: 0.0ss
Epoch [3/20] 完成 | 平均损失: 0.37300] | 损失: 0.3300 | 单步耗时: 0.087s | 预计剩余: 0.0ss
Epoch [4/20] 完成 | 平均损失: 0.23850] | 损失: 0.2160 | 单步耗时: 0.088s | 预计剩余: 0.0ss
Epoch [5/20] 完成 | 平均损失: 0.19880] | 损失: 0.2448 | 单步耗时: 0.087s | 预计剩余: 0.0ss
Epoch [6/20] 完成 | 平均损失: 0.17830] | 损失: 0.1497 | 单步耗时: 0.088s | 预计剩余: 0.0ss
Epoch [7/20] 完成 | 平均损失: 0.16240] | 损失: 0.1476 | 单步耗时: 0.087s | 预计剩余: 0.0ss
Epoch [8/20] 完成 | 平均损失: 0.15430] | 损失: 0.1461 | 单步耗时: 0.088s | 预计剩余: 0.0ss
Epoch [9/20] 完成 | 平均损失: 0.14610] | 损失: 0.1313 | 单步耗时: 0.087s | 预计剩余: 0.0ss
Epoch [10/20] 完成 | 平均损失: 0.13710] | 损失: 0.1545 | 单步耗时: 0.087s | 预计剩余: 0.0ss
Epoch [11/20] 完成 | 平均损失: 0.13470] | 损失: 0.1282 | 单步耗时: 0.087s | 预计剩余: 0.0ss
Epoch [12/20] 完成 | 平均损失: 0.13400] | 损失: 0.1237 | 单步耗时: 0.088s | 预计剩余: 0.0ss
Epoch [13/20] 完成 | 平均损失: 0.12180] | 损失: 0.1093 | 单步耗时: 0.088s | 预计剩余: 0.0ss
Epoch [14/20] 完成 | 平均

In [18]:
seed = "凤霞命苦啊，"
text = generate_text(model, char2idx, idx2char, seed_text=seed, gen_len=80)
print(f"\n种子: '{seed}'")
print(f"生成: '{text}'")
print("-" * 50)


种子: '凤霞命苦啊，'
生成: '凤霞命苦啊，你也别怪我心狠，都是那畜生胡来才会有今天。<EOS>"说完丈人又转向我，喊道："凤霞就留给你们徐家，家珍肚里的孩子就是我们陈家的人啦。<EOS>"我娘站在一旁呜'
--------------------------------------------------
